# 02. Baseline 모델 비교 + 불균형 처리 비교
Train 5-fold CV

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from steel import (BalancedWeightClassifier, CLASS_ORDER, CV, LabelEncodedClassifier, Oversample, RS, SCORING,
                   Undersample, build_preprocessor, cached, discrete_idx, load_data, make_pipeline)
pd.set_option("display.width", 200); pd.set_option("display.precision", 4)
pd.set_option("display.unicode.east_asian_width", True)   # 한글 출력 정렬
from joblib import Parallel, delayed
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler

X_train, X_test, y_train, y_test = load_data()
FEAT = list(X_train.columns)

## 1. Baseline: 6개 모델

In [2]:
# 기본 설정, 불균형 처리 없음
BASELINES = {
    "LogisticRegression": make_pipeline(LogisticRegression(max_iter=5000, random_state=RS), FEAT, scale=True),
    "DecisionTree": make_pipeline(DecisionTreeClassifier(random_state=RS), FEAT),
    "RandomForest": make_pipeline(RandomForestClassifier(random_state=RS, n_jobs=1), FEAT),
    "GBM": make_pipeline(GradientBoostingClassifier(random_state=RS), FEAT),
    "XGBoost": make_pipeline(LabelEncodedClassifier(XGBClassifier(random_state=RS, n_jobs=1)), FEAT),
    "LightGBM": make_pipeline(LGBMClassifier(random_state=RS, n_jobs=1, verbose=-1), FEAT),
}
def cv_row(m):
    r = cross_validate(m, X_train, y_train, cv=CV, scoring=SCORING, return_train_score=True)
    return {"macro_f1": r["test_macro_f1"].mean(), "std": r["test_macro_f1"].std(), "accuracy": r["test_accuracy"].mean(),
            "precision": r["test_precision"].mean(), "recall": r["test_recall"].mean(), "train_f1": r["train_macro_f1"].mean()}
base = pd.DataFrame(Parallel(n_jobs=-1)(delayed(cv_row)(m) for m in BASELINES.values()), index=list(BASELINES))
base.sort_values("macro_f1", ascending=False)
# → 부스팅 계열 우세 (XGBoost 최고). 트리 모델은 Train F1 ≈ 1 → 과적합 경향

,macro_f1,std,accuracy,precision,recall,train_f1
XGBoost,0.8109,0.0308,0.7945,0.8352,0.7997,1.0000
LightGBM,0.8044,0.0360,0.7893,0.8266,0.7936,1.0000
GBM,0.7958,0.0106,0.7784,0.8125,0.7850,0.9897
RandomForest,0.7899,0.0312,0.7719,0.8196,0.7718,1.0000
DecisionTree,0.7207,0.0259,0.7075,0.7218,0.7288,1.0000
LogisticRegression,0.7191,0.0275,0.7132,0.7324,0.7156,0.7488


## 2. 불균형 처리 비교

In [3]:
# 처리 4가지 × 부스팅·배깅 4개 모델 (샘플링·가중치는 학습 fold 안에서만 적용)
smote = lambda: SMOTENC(categorical_features=discrete_idx(FEAT), sampling_strategy=Oversample(1), k_neighbors=5, random_state=RS)
rus = lambda: RandomUnderSampler(sampling_strategy=Undersample(1), random_state=RS)
MODELS = {   # 이름: (기본 모델 생성, 가중치 적용 모델 생성)
    "RandomForest": (lambda: RandomForestClassifier(random_state=RS, n_jobs=1), lambda: RandomForestClassifier(random_state=RS, n_jobs=1, class_weight="balanced")),
    "GBM": (lambda: GradientBoostingClassifier(random_state=RS), lambda: BalancedWeightClassifier(GradientBoostingClassifier(random_state=RS), alpha=1)),
    "XGBoost": (lambda: LabelEncodedClassifier(XGBClassifier(random_state=RS, n_jobs=1)),
                lambda: BalancedWeightClassifier(LabelEncodedClassifier(XGBClassifier(random_state=RS, n_jobs=1)), alpha=1)),
    "LightGBM": (lambda: LGBMClassifier(random_state=RS, n_jobs=1, verbose=-1), lambda: LGBMClassifier(random_state=RS, n_jobs=1, verbose=-1, class_weight="balanced")),
}
def build(model, method):
    plain, weighted = MODELS[model]
    sampler = {"SMOTENC (균등화)": smote(), "언더샘플링 (균등화)": rus()}.get(method, "passthrough")
    est = weighted() if method == "클래스 가중치" else plain()
    return ImbPipeline([("prep", build_preprocessor(FEAT)), ("sampler", sampler), ("model", est)])

METHODS = ["처리 없음", "클래스 가중치", "SMOTENC (균등화)", "언더샘플링 (균등화)"]
splits = list(CV.split(X_train, y_train))
def fold(model, method, k):
    tr, va = splits[k]
    m = build(model, method).fit(X_train.iloc[tr], y_train.iloc[tr])
    return model, method, k, m.predict(X_train.iloc[va])
out = cached("imbalance_oof", lambda: Parallel(n_jobs=-1)(delayed(fold)(m, t, k) for m in MODELS for t in METHODS for k in range(5)))

oof, f1 = {}, {}
for m, t, k, p in out:
    oof.setdefault((m, t), pd.Series(index=y_train.index, dtype=object)).iloc[splits[k][1]] = p
    f1.setdefault((m, t), []).append(f1_score(y_train.iloc[splits[k][1]], p, average="macro"))
pd.DataFrame({t: {m: np.mean(f1[(m, t)]) for m in MODELS} for t in METHODS})
# → 언더샘플링은 데이터가 1/5로 줄어 크게 악화. LightGBM은 SMOTENC, GBM은 클래스 가중치가 가장 효과적

,처리 없음,클래스 가중치,SMOTENC (균등화),언더샘플링 (균등화)
RandomForest,0.7899,0.7853,0.7955,0.6664
GBM,0.7958,0.8092,0.8084,0.6764
XGBoost,0.8109,0.8091,0.8201,0.6638
LightGBM,0.8044,0.8150,0.8268,0.6811


## 3. 유형별 Recall 변화

In [4]:
# 소수 유형을 더 잡는 대신 Other_Faults를 더 놓침 (판정 경계 이동)
rec = pd.DataFrame({f"{m} / {t}": recall_score(y_train, oof[(m, t)], labels=CLASS_ORDER, average=None)
                    for m, t in [("GBM", "처리 없음"), ("GBM", "클래스 가중치"), ("LightGBM", "처리 없음"), ("LightGBM", "SMOTENC (균등화)")]}, index=CLASS_ORDER)
rec

,GBM / 처리 없음,GBM / 클래스 가중치,LightGBM / 처리 없음,LightGBM / SMOTENC (균등화)
Other_Faults,0.7528,0.6710,0.7639,0.7454
Bumps,0.6604,0.7290,0.6822,0.7134
K_Scratch,0.9553,0.9521,0.9585,0.9681
Z_Scratch,0.9145,0.9408,0.9276,0.9342
Pastry,0.5238,0.6905,0.5238,0.6429
Stains,0.8966,0.9310,0.9310,0.9655
Dirtiness,0.7955,0.8182,0.7727,0.7955
